#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


C:\Users\Raj Kumar\AppData\Local\Temp\ipykernel_2340\730491859.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct

In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

In [4]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [5]:
db = InMemoryVectorStore(embedding=embeddings)

In [7]:
db.add_documents(docs)

print("Vector Store Created Successfully!")

Vector Store Created Successfully!


In [8]:
query = "Why did America decide to enter the war?"

results = db.similarity_search(query, k=2)

for i, result in enumerate(results):
    print(f"\nResult {i+1}:")
    print(result.page_content)


Result 1:
To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.

Result 2:
To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.


In [10]:
# Initialize Hugging Face Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4566.16it/s]


In [12]:
db = InMemoryVectorStore(embedding=embeddings)

db.add_documents(docs)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [13]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
docs[0].page_content


'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [14]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [15]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='192347c3-eda5-42ab-9c31-f38cdad8d66e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  0.34645375515421756),
 (Document(id='05b9563b-f893-4990-aa24-98dc613e3df8', metadata={'source': 'spe

In [16]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.049689605832099915,
 0.10655157268047333,
 0.0015430882340297103,
 -0.013194373808801174,
 -0.04438777640461922,
 0.058573175221681595,
 0.04666462540626526,
 -0.02453066036105156,
 0.00482936529442668,
 0.020339909940958023,
 -0.03410932421684265,
 0.03873791918158531,
 0.01661890558898449,
 -0.018834881484508514,
 -0.007143031805753708,
 0.006980105768889189,
 0.02085006795823574,
 -0.035205330699682236,
 0.004866439383476973,
 -0.007269791793078184,
 0.07890860736370087,
 0.056077100336551666,
 0.06984832137823105,
 0.01655666157603264,
 0.013394526205956936,
 -0.012157352641224861,
 -0.020828312262892723,
 0.052774880081415176,
 -0.020383493974804878,
 0.026147596538066864,
 0.04590854048728943,
 -0.01753469929099083,
 0.016683507710695267,
 0.0021405683364719152,
 0.006523000076413155,
 0.0013917286414653063,
 -0.05417172610759735,
 -0.01658141054213047,
 -0.01163152139633894,
 -0.02048136293888092,
 -0.052918728440999985,
 -0.03948398306965828,
 0.0165480338037014,
 0.0405348

In [17]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='192347c3-eda5-42ab-9c31-f38cdad8d66e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='05b9563b-f893-4990-aa24-98dc613e3df8', metadata={'source': 'speech.txt'}, page_content='T

In [20]:
# Save vector database
db.dump("vector_store.json")

print("Vector Store Saved Successfully!")

Vector Store Saved Successfully!


In [21]:
# Load saved vector database
db = InMemoryVectorStore.load(
    "vector_store.json",
    embeddings
)

print("Vector Store Loaded Successfully!")

Vector Store Loaded Successfully!


In [22]:
query = "What is machine learning?"

result = db.similarity_search(query, k=3)

for doc in result:
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("-------------------------")

Content: It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.
Metadata: {'source': 'speech.txt'}
-------------------------
Content: To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of 